# Classification on a 2D toy dataset

**Relative entropy** 

* is also called Kullback-Leiber divergence.

* It measures the closeness of two probability distributions.

* Definition: Let $p=(p_1, p_2, \dots, p_C)$ and $q=(q_1, q_2, \cdots, q_C)$ be two probbility distributions over $\{1,2,\dots, C\}$.

$$ D_{KL}(p|q):= \sum_{i=1}^C \ln \frac{p_i}{q_i} p_i$$

**Tasks to verify**:

1.  Show that $D_{KL}(p|q)\ge 0$ and $D_{KL}(p|q)=0$ when $p=q$.
2.  Argue that in general $D_{KL}(p|q)\neq D_{KL}(q|p)$, i.e. it is not symmetric.
3.  Assume that $c_1, c_2, \cdots, c_N$ are sampled from $p$ and we want to approximate $p$ by $q=q(\theta)$. Then, as $N\rightarrow\infty$, minimizing $D_{KL}(p|q)$ 
    is equivalent to minimizing negative log-likelihood (NLL)
      $$ -\frac{1}{N}\sum_{i=1}^N\ln q_{c_i}$$
    

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import matplotlib.pyplot as plt

## Prepare the dataset

In [ ]:
N = 10000

# Gaussian data
x2d = torch.randn(N, 2)

# angle of each data point
angles = torch.atan2(x2d[:, 1], x2d[:,0])

print ("Range of angles: [%.2f, %.2f]" % (torch.min(angles), torch.max(angles)))

#number of classes
C = 6

# divide [-pi,pi] uniformly into C classes
d_theta = 2 * torch.pi / C
# class of each point
classes = torch.floor((angles - torch.min(angles)) / d_theta).to(dtype=torch.int64)

print ("Range of classes: [%d, %d]" % (torch.min(classes), torch.max(classes)))

# display the dataset
cm = plt.cm.get_cmap('RdYlBu')
plt.scatter(x2d[:,0], x2d[:,1], c=classes, cmap=cm)
plt.title("Dataset colored by classes")
plt.colorbar()
plt.show()

## Define a neural network model for our classifier

For each input data $x$, the probabilities of its class are

$$p_j(x;\theta) = \frac{e^{w_{j}}}{\sum_{c=1}^{C} e^{w_{c}}}, \quad j=1,2,\dots, C\,.$$.

The model learns to return the C-dim vector $w(x;\theta)=(w_{1}, w_2, \dots, w_C)$. 

* **Input**: 2d data of shape $[B, 2]$, $B$ is the batch-size.
* **Output**: shape $[B,C]$, where $w_{ij}=w^{(i)}_j=w_j(x_i;\theta)$ is the "energy" of the class $j=1,\dots, C$ corresponding to the $i$th input data $x_i$ for $i=1,\dots, B$.

**Note**: the map from $w$ to $p$ is called softmax: https://docs.pytorch.org/docs/2.12/generated/torch.nn.Softmax.html

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(2, 32)
        self.fc2 = nn.Linear(32, 32)
        # The output dimension has to match the number of classes
        self.fc3 = nn.Linear(32, C)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x        

## Visualize the untrained model

In [ ]:
model = Net()

output = model(x2d)

print ("Shape of model output:", output.shape)

# We simply take the index j of the largest w_j as the class predicted by the model.      
_, predicted = torch.max(output, 1)


fig, ax = plt.subplots(1,2, figsize=(8, 3))

cm = plt.cm.get_cmap('RdYlBu')

h0 = ax[0].scatter(x2d[:,0], x2d[:,1], c=classes, cmap=cm)
ax[0].set_title("True classes")

h1 = ax[1].scatter(x2d[:,0], x2d[:,1], c=predicted, cmap=cm)
ax[1].set_title("Classes predicted by untrained model")

plt.colorbar(h0)
plt.colorbar(h1)

plt.show()

## Training the model

The model is trained to minimize the loss:

$$ -\frac{1}{N} \sum_{i=1}^{N} \ln p^{(i)}_{c_i} = -\frac{1}{N} \sum_{i=1}^{N} \ln \frac{e^{w^{(i)}_{c_i}}}{\sum_{c=1}^{C} e^{w^{(i)}_{c}}}$$
where
* $c_i$: the class of $x_i$.
* $p^{(i)}_{j}= p_{j}(x_i;\theta)$, the probability that $x_i$ belongs to class $j$.
* $w^{(i)}_{j} = w_j(x_i;\theta)$.

**Task**: 
    Clarify the connection between the loss above for the classification task with the cross-entropy discussed at the very beginning of this notebook.

Reference: 
https://docs.pytorch.org/docs/2.12/generated/torch.nn.CrossEntropyLoss.html

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

n_epoch = 10
# batch-size
batch_size = 100

train_loader = torch.utils.data.DataLoader(torch.arange(N), batch_size=batch_size, shuffle=True, drop_last=True)

for epoch in range(n_epoch):
    running_loss = 0.0
    for i, indices in enumerate(train_loader):
        inputs = x2d[indices]
        labels = classes[indices]
        optimizer.zero_grad()
        output = model(inputs)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()
        # print statistics
        running_loss += loss.item()
        if i % 100 == 99:    
            print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 100:.3f}')
            running_loss = 0.0

## Predictions by the trained model. 

In [ ]:
output = model(x2d)
_, predicted = torch.max(output, 1)

print("Model output:\n", output)
print ("\nClasses predicted by model:\n", predicted)

## Prediction on a new dataset

Let's see how the model perform on a new dataset.


In [ ]:
x2d_new = torch.randn(N, 2)
output_new = model(x2d_new)
_, predicted_new = torch.max(output_new, 1)

In [ ]:
fig, ax = plt.subplots(1,3, figsize=(12, 3))

cm = plt.cm.get_cmap('RdYlBu')

h1=ax[0].scatter(x2d[:,0], x2d[:,1], c=classes, cmap=cm)
ax[0].set_title("True classes")

h2=ax[1].scatter(x2d[:,0], x2d[:,1], c=predicted, cmap=cm)
ax[1].set_title("Classes predicted by model")

h3=ax[2].scatter(x2d_new[:,0], x2d_new[:,1], c=predicted_new, cmap=cm)
ax[2].set_title("Classes of new data predicted by model")

plt.colorbar(h1)
plt.colorbar(h2)
plt.colorbar(h3)

plt.show()